### Complete end to end pipeline for ai authorizatioin and stylometric detection

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
df = pd.read_csv('./auth_datasets/authors_3dataset1.csv')
df.head()

Now removing html tags using regular expressions :::::

In [ ]:
import re

In [ ]:
def remove_html_tags(text): 
    clean = re.sub(r'<.*?>', '', text) 
    return clean

Data may be taken from multiple places and it may contains html tags

In [ ]:
df['content'].apply(remove_html_tags)

Now lets make some basic preprocessing::::>>>

In [ ]:
import nltk

In [ ]:
from nltk.tokenize import word_tokenize , sent_tokenize 
from nltk import pos_tag 
from collections import Counter 
import string

Stylometric frature extraction ::::

In [ ]:
def extract_feature(text): 
    words = word_tokenize(text) 
    sentences = sent_tokenize(text) 
    if(len(words)==0): return {} 
    tags = pos_tag(words) 
    pos_counts = Counter(tag for _, tag in tags) 
    features = { 
        "char_count": len(text), 
        "word_count": len(words), 
        "sent_count": len(sentences), 
        "avg_word_len" : sum(len(w) for w in words)/len(words), 
        "avg_sentence_len" : sum(len(sen) for sen in sentences)/len(sentences), 
        "unique_word_count" : len(set(words)), 
        "type_token_ratio" : len(set(words)) / len(words), 
        "noun_ratio" : pos_counts['NN']/len(tags), 
        "verb_ratio": pos_counts["VB"] / len(tags), 
        "adj_ratio": pos_counts["JJ"] / len(tags), 
        "adv_ratio": pos_counts["RB"] / len(tags), 
        'punctuation_count' : sum(1 for char in text if char in string.punctuation), 
        'upper_case_count' : sum(1 for word in words if word.isupper()), 
        } 
    
    for tag, count in pos_counts.items(): 
        features[f'pos_{tag}'] = count 
        ###-------------------------------------------------------------------------------------
        
    from nltk.corpus import stopwords 
    nltk.download('stopwords') 
    stop_words = set(stopwords.words('english')) 
    features['stopword_count'] = sum(1 for word in words if word.lower() in stop_words) 
    features['stopword_ratio'] = sum(1 for word in words if word.lower() in stop_words) / len(words) if len(words) > 0 else 0 
    return features

In [ ]:
stylometric_feat = df['content'].apply(extract_feature)

In [ ]:
stylometric_df = stylometric_feat.apply(pd.Series)

In [ ]:
stylometric_df = stylometric_df.fillna(0)

Vectorization of Content: Conversion of text to vector. For making the better model 
training and Stylometric detection 

### Here we use TfidfVectorizer

In [ ]:
### for conversion of text content to number so that we can apply model on it also 
## and can make stylometric detection 
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Creating Instance of TfidfVectorizer
vectorizer = vectorizer = TfidfVectorizer( 
    ngram_range=(1 , 2), 
    max_features=10000, 
    min_df=2 , 
    max_df=0.9 
    )

In [ ]:
# Converting df[content] into vector
X_ngram = vectorizer.fit_transform(df['content'].astype(str))

In [ ]:
# sparse matrix of dtype 'float64'
X_ngram

Here we have X_ngram as sparse matrix.
and 
stylometric_df as df with multiple features


Now We first requried to scale stylometric_df with standard scaler

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
scaled_stylometric_df = scaler.fit_transform(stylometric_df)

Now We have final scaled_stylometric_df ready to operate. 

But we have to convert it also into matrix to operate

In [ ]:
from scipy.sparse import csr_matrix
X_style_sparse = csr_matrix(scaled_stylometric_df)

In [ ]:
# Now combine it to final combined matrix ::::
from scipy.sparse import hstack 
final_df = hstack([X_ngram , X_style_sparse])

Here final_df contains the combined matrix

Now lets make the Train-Test split : 80% and 20% 

In [ ]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    final_df,
    df['author'],
    test_size=0.2,
    random_state=42
)

Making Label Encoding 

In [ ]:
from sklearn.preprocessing import LabelEncoder 
label_encoder = LabelEncoder() 
y_train = label_encoder.fit_transform(y_train) 
y_test = label_encoder.transform(y_test)

In [ ]:
## Now lets apply the model
from sklearn.ensemble import RandomForestClassifier 
model = RandomForestClassifier( 
    n_estimators=200, 
    random_state=42 
    )

In [ ]:
model.fit(X_train , y_train)

In [ ]:
# %% 
# from sklearn.linear_model import LogisticRegression 
# model = LogisticRegression
# (
# max_iter=1000, 
# random_state=42
# ) 
# model.fit(X_train , y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report 
print("Accuracy :", accuracy_score(y_test, y_pred)) 
print(classification_report(y_test, y_pred))

Thanking You :::>>>

Now lets save the model

In [ ]:
import joblib 
joblib.dump(model, "model.pkl") 
joblib.dump(scaler, "scaler.pkl") 
joblib.dump(label_encoder, "label_encoder.pkl") 
joblib.dump(vectorizer , "vectorizer.pkl")